# 영상 → 뎁스 영상 (컬러 / 흑백 / 알파) — Kaggle T4

직접 찍은 동작 영상이나 Blender 프리비즈를 **뎁스 영상**으로 바꿔요. 인물만 남기는 알파 뎁스도 만들 수 있어요.
MiniMax H3 같은 영상 모델에 **동작·카메라 레퍼런스**로 넣는 용도예요. 인물의 외형은 사라지고 움직임과 공간감만 남아요.

모델: [Video Depth Anything](https://github.com/DepthAnything/Video-Depth-Anything) (프레임 사이 깜빡임이 적은 영상 전용 뎁스 모델), 알파 계열은 [SAM 2](https://github.com/facebookresearch/sam2) (인물 추적 마스크)

**준비 (Kaggle 노트북 오른쪽 패널)**
1. Settings → Accelerator: **GPU T4** (x2 아니어도 돼요)
2. Settings → Internet: **On**
3. Add Input → 영상을 올린 **Dataset**을 추가하고, 설정 셀의 `INPUT_VIDEO` 경로를 맞춰요.

**실행:** 위에서부터 순서대로 실행해요. 결과는 `/kaggle/working/depth`에 저장돼요.

In [ ]:
# ============ 설정 (여기만 고치면 돼요) ============
INPUT_VIDEO = "/kaggle/input/my-clips/action.mp4"   # 직접 찍었거나 사용 권한이 있는 영상

START_SEC = 0.0          # 시작 지점(초)
MAX_SECONDS = 15.0       # 최대 길이(초). MiniMax H3 레퍼런스 영상은 2~15초
FPS = 24                 # MiniMax H3 기준 24fps
MAX_SIDE = 960           # 긴 변 최대 픽셀

DEPTH_MODEL = "large"    # "small" (빠름) | "base" | "large" (가장 정확, T4에서도 짧은 영상은 충분)
INPUT_SIZE = 518         # 모델 내부 처리 크기. 크게 하면 디테일이 늘고 느려져요 (14의 배수)
# 만들 뎁스 종류. 여러 개를 넣으면 각각 따로 저장해요.
#   전체 화면:  "color" 컬러 뎁스 (inferno, 보라-주황) | "turbo" 무지개색 컬러 뎁스 | "white" 흑백 뎁스 (가까울수록 흰색)
#   인물만 (SAM 2로 인물을 따라가며 배경을 검게 지워요):
#               "alpha_white" 알파 뎁스 (인물만 흑백 뎁스) | "alpha_color" 인물만 컬러 뎁스 | "alpha" 알파 매트 (흰 인물, 검은 배경)
DEPTH_STYLES = ["color", "white", "alpha_white"]
NEAR_IS_BRIGHT = True    # True: 가까운 곳이 밝게. False면 반전

# 인물 선택 (알파 계열을 고른 경우에만 사용)
PERSON_POINTS = [(0.5, 0.45)]   # 첫 프레임에서 인물 위의 점 (가로, 세로 비율 0~1). 여러 사람이면 각자 위에 점을 찍어요
BACKGROUND_POINTS = []          # 인물이 아닌데 같이 잡히는 곳 (제외할 점)
SAM_MODEL = "facebook/sam2.1-hiera-small"   # 더 정확하게: "facebook/sam2.1-hiera-large"
MASK_EXPAND = 4                 # 마스크 가장자리를 바깥으로 늘리는 픽셀 수
KEEP_AUDIO = True        # 원본 소리를 뎁스 영상에 그대로 넣기 (레퍼런스 영상의 타이밍 참고용)

WORK = "/tmp/depth_work"
OUT_DIR = "/kaggle/working/depth"

## 1. 환경 확인과 설치

In [ ]:
import os, shutil, subprocess, sys

def sh(cmd):
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=True)

subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv"])
if not os.path.isfile(INPUT_VIDEO):
    print("⚠ 입력 영상을 찾을 수 없어요:", INPUT_VIDEO)
    for root, _, files in os.walk("/kaggle/input"):
        for fn in files[:20]:
            print("   ", os.path.join(root, fn))

VDA = "/tmp/Video-Depth-Anything"
if not os.path.isdir(VDA):
    sh(f"git clone --depth 1 https://github.com/DepthAnything/Video-Depth-Anything {VDA}")
# 저장소의 requirements.txt는 구버전 torch를 고정해서 Kaggle 환경을 망가뜨려요. 필요한 것만 설치해요.
sh(f"{sys.executable} -m pip install -q einops easydict imageio imageio-ffmpeg huggingface_hub")

USE_ALPHA = any(s.startswith("alpha") for s in DEPTH_STYLES)
if USE_ALPHA:
    # SAM 2: 의존성을 따로 설치하고 --no-deps로 넣어서 Kaggle의 torch를 건드리지 않아요
    sh(f"{sys.executable} -m pip install -q hydra-core iopath")
    sh(f"SAM2_BUILD_CUDA=0 {sys.executable} -m pip install -q --no-deps git+https://github.com/facebookresearch/sam2.git")

FFMPEG = shutil.which("ffmpeg")
if not FFMPEG:
    import imageio_ffmpeg
    FFMPEG = imageio_ffmpeg.get_ffmpeg_exe()

import torch
print("torch", torch.__version__, "| GPU", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음")

## 2. 영상 준비 (길이 자르기, 24fps, 크기 맞추기)

In [ ]:
import glob, os, shutil, subprocess
import cv2
import numpy as np

shutil.rmtree(WORK, ignore_errors=True)
os.makedirs(f"{WORK}/frames")
os.makedirs(OUT_DIR, exist_ok=True)

scale = (f"scale='if(gte(iw,ih),min({MAX_SIDE},iw),-2)':'if(gte(iw,ih),-2,min({MAX_SIDE},ih))'"
         ",scale=trunc(iw/2)*2:trunc(ih/2)*2")
trim = ["-ss", str(START_SEC), "-t", str(MAX_SECONDS)]
subprocess.run([FFMPEG, "-y", "-loglevel", "error", *trim, "-i", INPUT_VIDEO,
                "-vf", f"fps={FPS},{scale}", "-q:v", "2", "-start_number", "0",
                f"{WORK}/frames/%05d.jpg"], check=True)

AUDIO = None
if KEEP_AUDIO:
    r = subprocess.run([FFMPEG, "-y", "-loglevel", "error", *trim, "-i", INPUT_VIDEO,
                        "-vn", "-c:a", "aac", "-b:a", "192k", f"{WORK}/audio.m4a"])
    if r.returncode == 0 and os.path.getsize(f"{WORK}/audio.m4a") > 0:
        AUDIO = f"{WORK}/audio.m4a"
    else:
        print("소리가 없는 영상이에요. 소리 없이 진행해요.")

paths = sorted(glob.glob(f"{WORK}/frames/*.jpg"))
frames = np.stack([cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB) for p in paths])
H, W = frames.shape[1:3]
print(f"{len(frames)}프레임 ({len(frames) / FPS:.1f}초), {W}x{H}")

## 3. 뎁스 추출

In [ ]:
import gc, sys
import torch
from huggingface_hub import hf_hub_download

sys.path.insert(0, VDA)
from video_depth_anything.video_depth import VideoDepthAnything

CONFIGS = {
    "small": ("depth-anything/Video-Depth-Anything-Small", "vits", {"features": 64, "out_channels": [48, 96, 192, 384]}),
    "base": ("depth-anything/Video-Depth-Anything-Base", "vitb", {"features": 128, "out_channels": [96, 192, 384, 768]}),
    "large": ("depth-anything/Video-Depth-Anything-Large", "vitl", {"features": 256, "out_channels": [256, 512, 1024, 1024]}),
}
repo, encoder, cfg = CONFIGS[DEPTH_MODEL]
ckpt = hf_hub_download(repo, f"video_depth_anything_{encoder}.pth")

model = VideoDepthAnything(encoder=encoder, **cfg)
model.load_state_dict(torch.load(ckpt, map_location="cpu", weights_only=True), strict=True)
model = model.to("cuda").eval()

# T4는 fp16으로 처리해요 (fp32=False)
depths, _ = model.infer_video_depth(frames, FPS, input_size=INPUT_SIZE, device="cuda", fp32=False)
depths = np.asarray(depths, dtype=np.float32)
print("뎁스", depths.shape, f"범위 {depths.min():.3f} ~ {depths.max():.3f}")

del model
gc.collect()
torch.cuda.empty_cache()

## 4. 인물 선택 확인 (알파 계열을 고른 경우)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

ALPHA = None
if not USE_ALPHA:
    print("알파 계열을 고르지 않아서 건너뛰어요.")
else:
    from sam2.sam2_video_predictor import SAM2VideoPredictor

    predictor = SAM2VideoPredictor.from_pretrained(SAM_MODEL, device="cuda")
    state = predictor.init_state(video_path=f"{WORK}/frames", offload_video_to_cpu=True)
    pts = [(x * W, y * H) for x, y in PERSON_POINTS] + [(x * W, y * H) for x, y in BACKGROUND_POINTS]
    labels = [1] * len(PERSON_POINTS) + [0] * len(BACKGROUND_POINTS)
    with torch.inference_mode():
        _, _, logits = predictor.add_new_points_or_box(
            state, frame_idx=0, obj_id=1,
            points=np.array(pts, dtype=np.float32), labels=np.array(labels, dtype=np.int32))
    first = (logits[0, 0] > 0).cpu().numpy()

    overlay = frames[0].copy()
    overlay[first] = (overlay[first] * 0.4 + np.array([255, 40, 40]) * 0.6).astype(np.uint8)
    plt.figure(figsize=(10, 10 * H / W))
    plt.imshow(overlay)
    for (x, y), lab in zip(pts, labels):
        plt.scatter([x], [y], c="lime" if lab else "red", s=120, marker="*" if lab else "x")
    plt.title("frame 0: red = person mask (green * = person point, red x = background point)")
    plt.axis("off")
    plt.show()
    print("빨간 영역이 인물로 잡힌 부분이에요. 틀리면 PERSON_POINTS / BACKGROUND_POINTS를 고치고 이 셀을 다시 실행하세요.")

## 5. 인물 마스크를 영상 전체로 따라가기

In [ ]:
import cv2
import numpy as np
import torch

if USE_ALPHA:
    masks = np.zeros((len(frames), H, W), dtype=bool)
    with torch.inference_mode():
        for idx, _, logits in predictor.propagate_in_video(state):
            masks[idx] = (logits > 0).any(dim=0)[0].cpu().numpy()
    if MASK_EXPAND > 0:
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * MASK_EXPAND + 1,) * 2)
        masks = np.stack([cv2.dilate(m.astype(np.uint8), kernel) > 0 for m in masks])
    # 가장자리를 살짝 부드럽게 (계단 현상 방지)
    ALPHA = np.stack([cv2.GaussianBlur(m.astype(np.float32), (5, 5), 0) for m in masks])

    area = masks.reshape(len(masks), -1).mean(axis=1) * 100
    empty = np.where(area == 0)[0]
    jumps = np.where(np.abs(np.diff(area)) > np.maximum(area[:-1], 1e-3) * 0.5)[0] + 1
    print(f"인물 면적: 평균 {area.mean():.1f}%, 최소 {area.min():.1f}%, 최대 {area.max():.1f}%")
    if len(empty):
        print(f"⚠ 인물이 사라진 프레임 {len(empty)}개: {(empty / FPS).round(2).tolist()[:20]}초")
    if len(jumps):
        print(f"⚠ 마스크가 갑자기 크게 바뀐 프레임 {len(jumps)}개: {(jumps / FPS).round(2).tolist()[:20]}초 (아래 결과에서 확인해 보세요)")

    del predictor, state
    torch.cuda.empty_cache()

## 6. 뎁스 영상 저장

In [ ]:
import os, subprocess, time
import cv2
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Video, display

# 영상 전체 기준으로 정규화해서 프레임마다 밝기가 튀지 않게 해요 (극단값 1%는 잘라냄)
lo, hi = np.percentile(depths, [1, 99])
norm = np.clip((depths - lo) / max(hi - lo, 1e-6), 0, 1)
if not NEAR_IS_BRIGHT:
    norm = 1 - norm
levels = (norm * 255).astype(np.uint8)
if levels.shape[1:3] != (H, W):
    levels = np.stack([cv2.resize(f, (W, H), interpolation=cv2.INTER_LINEAR) for f in levels])

def render(style):
    if style.startswith("alpha"):
        if ALPHA is None:
            raise RuntimeError("알파 계열은 4~5번 셀을 먼저 실행해야 해요.")
        if style == "alpha":
            return np.repeat((ALPHA * 255).astype(np.uint8)[..., None], 3, axis=-1)
        base = render(style.removeprefix("alpha_"))
        return (base * ALPHA[..., None]).astype(np.uint8)
    if style == "white":
        return np.repeat(levels[..., None], 3, axis=-1)
    cmap = {"color": "inferno", "turbo": "turbo"}[style]
    lut = (matplotlib.colormaps[cmap](np.linspace(0, 1, 256))[:, :3] * 255).astype(np.uint8)
    return lut[levels]

def encode(frames_rgb, path, audio=None):
    cmd = [FFMPEG, "-y", "-loglevel", "error", "-f", "rawvideo", "-pix_fmt", "rgb24",
           "-s", f"{frames_rgb.shape[2]}x{frames_rgb.shape[1]}", "-r", str(FPS), "-i", "-"]
    if audio:
        cmd += ["-i", audio, "-c:a", "copy", "-shortest"]
    cmd += ["-c:v", "libx264", "-pix_fmt", "yuv420p", "-crf", "16", path]
    p = subprocess.Popen(cmd, stdin=subprocess.PIPE)
    p.stdin.write(np.ascontiguousarray(frames_rgb).tobytes())
    p.stdin.close()
    if p.wait() != 0:
        raise RuntimeError(f"인코딩 실패: {path}")

name = os.path.splitext(os.path.basename(INPUT_VIDEO))[0]
stamp = time.strftime("%m%d_%H%M")
rendered = {style: render(style) for style in DEPTH_STYLES}
for style, video in rendered.items():
    path = f"{OUT_DIR}/{name}_depth_{style}_{stamp}.mp4"
    encode(video, path, AUDIO)
    print("저장:", path)
SIDE_MP4 = f"{OUT_DIR}/{name}_compare_{stamp}.mp4"
# 원본 | 뎁스들 나란히 (확인용). 여러 개면 전체 폭이 2880px을 넘지 않게 줄여요
panels = [frames, *rendered.values()]
k = min(1.0, 2880 / (W * len(panels)))
pw, ph = int(W * k) // 2 * 2, int(H * k) // 2 * 2
side = np.concatenate([p if k == 1.0 else np.stack([cv2.resize(f, (pw, ph), interpolation=cv2.INTER_AREA) for f in p])
                       for p in panels], axis=2)
encode(side, SIDE_MP4)
print("저장:", SIDE_MP4)

# 전체 구간 확인용 썸네일 8장
idx = np.linspace(0, len(frames) - 1, 8).astype(int)
rows = [frames, *rendered.values()]
fig, axes = plt.subplots(len(rows), 8, figsize=(20, len(rows) * 2.5 * H / W + 1), squeeze=False)
for r, row in enumerate(rows):
    for col, i in enumerate(idx):
        axes[r, col].imshow(row[i]); axes[r, col].axis("off")
        if r == 0:
            axes[r, col].set_title(f"{i / FPS:.1f}s")
plt.tight_layout()
plt.show()
display(Video(SIDE_MP4, embed=True, width=900))

## 다음 단계: MiniMax H3에 레퍼런스 영상으로 쓰기

1. 오른쪽 위 **Save Version**으로 노트북을 저장하면 `/kaggle/working/depth`의 결과가 남아요.
2. Singularity 노트북에서 **Add Input → Notebook Output**으로 이 노트북을 추가해요.
3. 프롬프트에서 동작은 `<Video 1>`, 외형은 `<Picture 1>`에서 가져오라고 적어요. 예:
   `<Subject 1> is the old man whose motion comes from <Video 1> and whose appearance comes from <Picture 1>.`

**팁**
- 알파 계열에서 인물이 잘못 잡히면 4번 셀의 미리보기를 보면서 `PERSON_POINTS`(인물 위)와 `BACKGROUND_POINTS`(제외할 곳)를 고쳐요. 점은 화면 비율 좌표라서 (0.5, 0.5)가 정가운데예요.
- 뎁스가 흐릿하면 `DEPTH_MODEL="large"`, `INPUT_SIZE=644`처럼 올려 보세요.
- 촬영할 때 배경이 단순하고 인물과 거리가 분명하면 뎁스가 깔끔하게 나와요.
- 오류가 나면 메시지를 그대로 복사해서 Claude에게 붙여 주세요.